# ML-03 — Frame Your Lane as an ML Task

This notebook frames the **Refresh / Content Opportunity Scoring** lane as a ranking problem. The goal is to prioritize pages for human review; it is not to claim that the model predicts Google’s algorithm.

## 1. My lane as an ML task (type)

**Lane:** Refresh / Content Opportunity Scoring.

**Task type:** Ranking / scoring, with a classification-style proxy used to train the ranker. The decision is: out of many pages, which pages should an editor review first? The model will produce a priority score and pages will be sorted from highest to lowest priority.

**Who acts:** A content editor or SEO reviewer.

**Action:** Review the highest-ranked pages and decide whether to refresh, rewrite the title or metadata, expand the content, or leave the page on a monitoring list.

**Cost of a wrong call:** A false positive uses editorial time on a page that did not need work. A false negative leaves a declining or underperforming page low in the queue and may delay a useful refresh.

In [ ]:
lane = 'Refresh / Content Opportunity Scoring'
task_type = 'ranking / scoring'
decision_owner = 'Content strategist / SEO editor'
action = 'Review and refresh the highest-priority pages first'
print('Lane:', lane)
print('Task type:', task_type)
print('Decision owner:', decision_owner)
print('Action:', action)

## 2. Target or proxy

**Prediction target:** Whether a page is currently showing a declining search trend.

**Proxy label:** `is_declining_label = (trend_direction == 'down')`. This is a rule-derived label rather than a measured future outcome. It gives us a consistent target for comparing a baseline and a model.

**Leakage control:** `trend_direction` and `trend_pct` will not be used as model features because they directly define or closely expose the label. `content_id` and `client_id` are pseudonymous identifiers and will be used only for grouping and client-level validation, never as features.

In [ ]:
import pandas as pd
from pathlib import Path

candidate_paths = [Path('data/raw/content_refresh_anonymized.csv'), Path('../data/raw/content_refresh_anonymized.csv'), Path('../../data/raw/content_refresh_anonymized.csv')]
csv_path = next((path for path in candidate_paths if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError('Could not locate data/raw/content_refresh_anonymized.csv. Run from the repository or clone it first.')

df = pd.read_csv(csv_path)
df['is_declining_label'] = df['trend_direction'].astype('string').str.lower().eq('down').astype('int8')
leakage_columns = {'trend_direction', 'trend_pct', 'is_declining_label'}
candidate_features = ['search_volume','competition','cpc','word_count','char_count','impressions_90d','clicks_90d','sessions_90d','content_age_days','days_since_last_update','ctr','avg_position','engagement_rate','scroll_rate','ai_traffic_pct','content_type','main_intent']
safe_features = [column for column in candidate_features if column not in leakage_columns]
print('Rows:', f'{len(df):,}')
print('Declining pages:', f"{df['is_declining_label'].sum():,}")
print('Declining rate:', f"{df['is_declining_label'].mean():.1%}")
print('Leakage columns excluded:', sorted(leakage_columns.intersection(safe_features)))
print('Number of safe candidate features:', len(safe_features))

## 3. Success metric

The primary metric is **Precision@50**: among the 50 pages at the top of the ranked queue, what fraction are labeled as declining? This matches the real decision because an editor has limited review capacity.

I will compare Precision@50 with the overall declining base rate and compare the model with a transparent rule baseline on the same evaluation split.

In [ ]:
def precision_at_k(y_true, scores, k=50):
    frame = pd.DataFrame({'label': list(y_true), 'score': list(scores)})
    top_k = frame.sort_values('score', ascending=False).head(k)
    return float(top_k['label'].mean()) if len(top_k) else 0.0

base_rate = float(df['is_declining_label'].mean())
print('Primary metric: Precision@50')
print('Base rate:', f'{base_rate:.3f}')
print('A model should beat the base rate in the top-50 review queue.')

## 4. The unit of analysis, as a real dataframe

One row represents one pseudonymous content page for one client, summarized over the available trailing 90-day measurement window. The model ranks pages, so the prediction unit and the action unit are both the content page.

In [ ]:
id_columns = [column for column in ['content_id', 'client_id'] if column in df.columns]
display_columns = id_columns + ['content_type', 'impressions_90d', 'avg_position', 'trend_direction', 'is_declining_label']
print('One row = one content page.')
print('Rows:', f'{len(df):,}')
print('Unique content pages:', f"{df['content_id'].nunique():,}")
print('Unique clients:', f"{df['client_id'].nunique():,}")
display(df[display_columns].head())

## 5. Why ML beats a fixed rule here

A single rule such as ‘refresh every page with low CTR’ cannot combine the many signals that influence review priority. Pages differ in traffic volume, position, age, content type, engagement, freshness, and search intent. A model can learn a more flexible combination of these signals and produce a ranked queue.

The model still needs to be compared with a transparent rule baseline. The purpose is not to replace editorial judgment or prove causality; it is to make the first-pass review queue more consistent and evidence-based.

In [ ]:
signal_columns = ['impressions_90d', 'avg_position', 'content_age_days', 'days_since_last_update', 'ctr', 'engagement_rate']
available_signals = [column for column in signal_columns if column in df.columns]
display(df[available_signals].describe().T[['count', 'mean', 'min', 'max']])

## Self-check

- [x] The task is framed as a ranking/scoring problem.
- [x] The target and proxy definition are explicit.
- [x] Precision@50 is connected to the review decision.
- [x] The unit of analysis is verified with the dataframe.
- [x] Label-derived fields and pseudonymous IDs are excluded from modeling features.
- [x] The notebook contains written answers and executable code.